### LLM_Project

In [ ]:
!pip install google-generativeai         #Google’s generative models (Gemini API)
!pip install datasets                    # Hugging Face’s library for loading, preprocessing, and sharing datasets for NLP, vision,etc
!pip install -U bitsandbytes             # CUDA-based library for 8-bit and 4-bit quantization and optimizers to reduce memory usage
!pip install transformers                # pretrained transformer models (LLMs)
!pip install -U peft                     # Parameter-Efficient Fine-Tuning - HF library for LoRA and other PEFT methods
!pip install -U "huggingface_hub[cli]"   # client + command-line interface (hf / huggingface-cli) to interact with the Hugging Face Hub
!pip install -U trl                      # HF library for post-training LLMs using SFT, DPO, GRPO, PPO

In [3]:
pip install --upgrade scipy scikit-learn

  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 64.5 MB/s  0:00:00
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/2 [scipy]
    Found existing installation: scikit-learn 1.5.232m0/2 [scipy]
    Uninstalling scikit-learn-1.5.2:━━━━━━━━ 0/2 [scipy]
      Successfully uninstalled scikit-learn-1.5.20/2 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


### Extracting and cleaning the text-book from project Gutenberg website

In [5]:
# we extract 3 books in different subjects: physics, biology, and economics

import requests
import os

books = {
    "physics_common_science": {
        "subject": "physics",
        "title": "Common Science",
        "url": "https://www.gutenberg.org/ebooks/29838.txt.utf-8"
    },
    "biology_civic_biology": {
        "subject": "biology",
        "title": "A Civic Biology, Presented in Problems",
        "url": "https://www.gutenberg.org/ebooks/39969.txt.utf-8"
    },
    "economics_principles_fetter": {
        "subject": "economics",
        "title": "The Principles of Economics, with Applications to Practical Problems",
        "url": "https://www.gutenberg.org/ebooks/40077.txt.utf-8"
    }
}

os.makedirs("gutenberg_books", exist_ok=True)

clean_texts = {}

for key, info in books.items():
    print(f"\n ++ {info['title']} ({info['subject']}): ++")

    #  download 
    response = requests.get(info["url"])
    response.raise_for_status()
    response.encoding = "utf-8"
    text = response.text

    # strip Gutenberg header/footer inline 
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker   = "*** END OF THE PROJECT GUTENBERG EBOOK"

    start_idx = text.find(start_marker)
    if start_idx != -1:
        start_idx = text.find("\n", start_idx)  # move to end of that line
    else:
        start_idx = 0

    end_idx = text.find(end_marker)
    if end_idx == -1:
        end_idx = len(text)

    clean_text = text[start_idx:end_idx].strip()

    # store books
    filename = f"gutenberg_books/{key}_clean.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(clean_text)

    clean_texts[key] = clean_text
    print(f"Saved cleaned text to: {filename}")
    print(f"Characters (cleaned): {len(clean_text):,}")



 ++ Common Science (physics): ++
Saved cleaned text to: gutenberg_books/physics_common_science_clean.txt
Characters (cleaned): 576,822

 ++ A Civic Biology, Presented in Problems (biology): ++
Saved cleaned text to: gutenberg_books/biology_civic_biology_clean.txt
Characters (cleaned): 799,101

 ++ The Principles of Economics, with Applications to Practical Problems (economics): ++
Saved cleaned text to: gutenberg_books/economics_principles_fetter_clean.txt
Characters (cleaned): 1,240,546


In [6]:
for key, info in books.items():
    print(f"\n SAMPLE FROM from {info['title']} ({key}) ")
    sample = clean_texts[key][:100]
    print(sample)
    print("\n" + "="*80)


 SAMPLE FROM from Common Science (physics_common_science) 
Produced by David Garcia, Simon Gardner and the Online
Distributed Proofreading Team at https://www


 SAMPLE FROM from A Civic Biology, Presented in Problems (biology_civic_biology) 
Produced by Mark C. Orton, Carol Ann Brown, and the Online
Distributed Proofreading Team at http://


 SAMPLE FROM from The Principles of Economics, with Applications to Practical Problems (economics_principles_fetter) 
THE PRINCIPLES OF ECONOMICS

WITH APPLICATIONS TO PRACTICAL PROBLEMS

BY

FRANK A. FETTER, PH.



In [ ]:
### Interacting with Gemini to make 30 question for each book

In [7]:
import google.generativeai as genai
import os
import re
import pandas as pd

os.environ["GEMINI_API_KEY"] = 
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

In [8]:
import re
import pandas as pd

books = [
    ("physics_common_science",   "physics"),
    ("biology_civic_biology",    "biology"),
    ("economics_principles_fetter", "economics")
]

num_questions = 30
chunk_length = 40000

all_entries = []

for key, subject in books:
    print(f"\n=== Processing {key} ({subject}) ===")

    text = clean_texts[key]

    # we select some part of each books
    idx = text.find("What a plant takes from the soil and how it gets it.") # for biology chapther VI
    if idx == -1:
        idx = text.find("SECTION 4.") # for physics
    if idx == -1:
        idx = text.find("CHAPTER 4") # for economics
    if idx == -1:
        print("  Could not find CHAPTER , using index 0 as fallback")
        idx = 0

    start = idx
    end = start + chunk_length
    if end > len(text):
        end = len(text)

    chunk = text[start:end]
    print(f"  Using chunk from {start} to {end}, length = {end - start}")

    # defining Gemini prompt to write quaestions
    prompt = f"""
You are helping me build a textbook-based question bank.

SUBJECT: {subject.capitalize()}
TEXTBOOK EXCERPT:
\"\"\"{chunk}\"\"\"

TASK:
1. Read the excerpt carefully.
2. Create {num_questions} diverse, high-quality question–answer pairs
   that a university or advanced high school student could answer using
   ONLY the information in this excerpt.
3. Mix question types: conceptual, definitions, reasoning, applications,
   and a few that require combining information from different parts.
4. Avoid trivial questions that ask about single random sentences.
5. Each question must be fully self-contained.
6. Do NOT say things like "based on the text", "according to this passage",
   "from the excerpt",  "from the text", or "in the above text".
7. Focus on the underlying subject concepts ( definitions, laws, relationships, concepts),
   NOT on the book's preface, pedagogy, or how chapters are organized.
8. Do NOT ask question that need to page numbers, sections, text, or the existence of the textbook.

FORMAT:
Return the result as plain text in the following structure:

Q1: <question text>
A1: <short but complete correct answer>

Q2: <question text>
A2: <answer>

...

Q{num_questions}: <question text>
A{num_questions}: <answer>
"""

    response = model.generate_content(prompt)
    qa_block = response.text

    print("  Preview:\n", qa_block[:500], "\n")

    # parse Qn / An into entries 
    lines = qa_block.splitlines()
    current_q = None
    current_a = None
    q_index = None

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Q<number>:
        m_q = re.match(r"^Q(\d+):\s*(.*)", line)
        if m_q:
            # save previous pair if exists
            if current_q is not None and current_a is not None:
                all_entries.append({
                    "book_key": key,
                    "subject": subject,
                    "q_index": q_index,
                    "question": current_q.strip(),
                    "answer": current_a.strip()
                })
            q_index = int(m_q.group(1))
            current_q = m_q.group(2)
            current_a = ""
            continue

        # A<number>:
        m_a = re.match(r"^A(\d+):\s*(.*)", line)
        if m_a:
            current_a = m_a.group(2)
            continue

        # continuation of answer
        if current_a is not None:
            current_a += " " + line

    # save last pair
    if current_q is not None and current_a is not None:
        all_entries.append({
            "book_key": key,
            "subject": subject,
            "q_index": q_index,
            "question": current_q.strip(),
            "answer": current_a.strip()
        })

print("\nTotal Q&A pairs collected:", len(all_entries))



=== Processing physics_common_science (physics) ===
  Using chunk from 52345 to 92345, length = 40000
  Preview:
 Q1: Explain why an object placed in water in a gravity-free environment does not float to the surface or sink.
A1: In a gravity-free environment, there is no "up" or "down" and no gravitational pull. Therefore, an object like a nail or cork would simply stay where it is placed in the water because it has no weight, and the water itself also has no weight, so there is no gravitational force to pull the water under the object and push it up or aside.

Q2: What fundamental condition must be met for 


=== Processing biology_civic_biology (biology) ===
  Using chunk from 120686 to 160686, length = 40000
  Preview:
 Q1: What is the primary function of the root for a young seed plant?
A1: One of the most important functions of the root to a young seed plant is that of a holdfast, an anchor to fasten it in the place where it is to develop.

Q2: Besides anchoring the plant, list t

In [22]:
df_qa = pd.DataFrame(all_entries)
df_qa.head()
df_qa.to_csv("qa_gemini_all_subjects.csv", index=False)

### making a test set for base model (Mistral-7B-Instruct-v0.2)

In [1]:
# make a test set for intial interacting to evaluate in which subject is weaker
import pandas as pd
df_qa = pd.read_csv("qa_gemini_all_subjects.csv")
df_test = df_qa.copy()
len(df_test), df_test.head()

(90,
                  book_key  subject  q_index  \
 0  physics_common_science  physics        1   
 1  physics_common_science  physics        2   
 2  physics_common_science  physics        3   
 3  physics_common_science  physics        4   
 4  physics_common_science  physics        5   
 
                                             question  \
 0  What are molecules, and how do scientists know...   
 1  How does a thermometer work to indicate temper...   
 2  Explain the molecular reason why heat causes o...   
 3  What is the definition of heat according to th...   
 4  Describe the process by which an object can co...   
 
                                               answer  
 0  Molecules are tiny moving specks that make up ...  
 1  The mercury in the thermometer's bulb expands ...  
 2  When an object is heated, its molecules move f...  
 3               Heat is the motion of the molecules.  
 4  If a thing expands without being heated from a...  )

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, \
    BitsAndBytesConfig, TrainingArguments, pipeline, logging

In [14]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# we have only test part, we don't configure any train or validation part.

# 4-bit quantization to fit on GPU more easily
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:08<00:00,  2.88s/it]
Device set to use cuda:0


In [19]:
import re
from tqdm import tqdm  

results = []

for i, row in tqdm(df_test.iterrows(), total=len(df_test)):
    subject = row["subject"]
    question = row["question"]
    gold_answer = row["answer"]

    # prompt for Mistral-7B-Instruct
    prompt = (
        "<s>[INST] You are an expert {subject} tutor "
        + " Answer the following question in ONE short sentence that is fully correct and directly answers the question.\n\n"
        + "Question: " + question + " [/INST]"
    )

    out = pipe(
        prompt,
        max_new_tokens=32,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    # removing the prompt part, we keep only the model's continuation
    model_answer = out[len(prompt):].strip()

    results.append({
        "subject": subject,
        "question": question,
        "gold_answer": gold_answer,
        "model_answer": model_answer,
    })

df_results = pd.DataFrame(results)
df_results.to_csv("mistral_base_answers_before_finetune.csv", index=False)
df_results.head()


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 90/90 [01:10<00:00,  1.28it/s]


,subject,question,gold_answer,model_answer
0,physics,"What are molecules, and how do scientists know...",Molecules are tiny moving specks that make up ...,"Molecules are fundamental units of matter, and..."
1,physics,How does a thermometer work to indicate temper...,The mercury in the thermometer's bulb expands ...,A thermometer works by expanding or contractin...
2,physics,Explain the molecular reason why heat causes o...,"When an object is heated, its molecules move f...",Heat causes objects to expand because it incre...
3,physics,What is the definition of heat according to th...,Heat is the motion of the molecules.,"Heat is defined as energy in motion, specifica..."
4,physics,Describe the process by which an object can co...,If a thing expands without being heated from a...,An object can cool down through expansion by r...


In [25]:
gemini_scores = []

for i, row in df_results.iterrows():
    question = row["question"]
    gold = row["gold_answer"]
    pred = row["model_answer"]

    judge_prompt = f"""
You are grading a student's answer to a question, using a reference answer.

QUESTION:
{question}

REFERENCE ANSWER (ideal, but not the only correct phrasing):
{gold}

STUDENT ANSWER:
{pred}

TASK:
1. Compare the student answer to the reference answer.
2. Judge ONLY the factual correctness and completeness, not wording style.
3. Ignore small wording differences if the meaning is the same.
4. Give a single integer SCORE from 1 to 5:
   - 5 = completely correct and essentially as good as the reference
   - 4 = mostly correct, maybe missing a small detail
   - 3 = partially correct, but missing important points or having minor errors
   - 2 = mostly incorrect, with some small correct pieces
   - 1 = wrong, irrelevant, or nonsensical

FORMAT:
Just answer with the number 1, 2, 3, 4, or 5. Do not add any explanation.
"""

    score_resp = model.generate_content(judge_prompt)
    raw_score = score_resp.text.strip()
    # try to parse to int safely
    try:
        score = int(raw_score)
    except:
        # if Gemini says "Score: 4", extract digit
        m = re.search(r"[1-5]", raw_score)
        score = int(m.group(0)) if m else None

    gemini_scores.append(score)

df_results["llm_score"] = gemini_scores


In [26]:
print("Overall Gemini-judge average score:", df_results["llm_score"].mean())

Overall Gemini-judge average score: 3.1666666666666665


In [27]:
print(df_results.groupby("subject")["llm_score"].mean())

subject
biology      2.833333
economics    3.166667
physics      3.500000
Name: llm_score, dtype: float64
